# TISER Dataset Chunker for Manual Translation

This notebook is designed to support a **human-in-the-loop translation process**
for the TISER dataset.

The goal is to:
- Load a large JSON dataset (train or test)
- Split it into manageable chunks
- Display **one chunk at a time**
- Allow manual translation using ChatGPT / Gemini
- Avoid confusion or loss of alignment between chunks

Each execution of the main cell will output **exactly one chunk**.

## Configuration

Set:
- the input dataset path
- the chunk size
- the progress file used to remember which chunk was last shown

The progress file ensures that you can safely stop and resume the process
without reprocessing the same examples.

In [ ]:
import json
from pathlib import Path

# ===== USER CONFIG =====
DATASET_PATH = Path("data/raw/TISER_train.json")   # change to train or test
CHUNK_SIZE = 10                                    # examples per chunk
PROGRESS_FILE = Path(".translation_progress.json") # internal state
# =======================

## Load Dataset

The dataset is expected to be a JSON list where each element corresponds
to one TISER example.

No modification is applied at this stage.

In [ ]:
with DATASET_PATH.open("r", encoding="utf-8") as f:
    dataset = json.load(f)

total_examples = len(dataset)
total_chunks = (total_examples + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f"Loaded dataset: {DATASET_PATH}")
print(f"Total examples: {total_examples}")
print(f"Chunk size: {CHUNK_SIZE}")
print(f"Total chunks: {total_chunks}")

## Load or Initialize Progress

The progress file stores the index of the **next chunk to display**.
This allows incremental translation without duplication.

In [ ]:
if PROGRESS_FILE.exists():
    with PROGRESS_FILE.open("r", encoding="utf-8") as f:
        progress = json.load(f)
    current_chunk_idx = progress.get("current_chunk", 0)
else:
    current_chunk_idx = 0

print(f"Current chunk index: {current_chunk_idx}/{total_chunks}")

## Display Next Chunk

Running this cell will:
- Extract the next chunk
- Print it as formatted JSON
- Update the progress file

Copy the output JSON and paste it into ChatGPT / Gemini for translation.

In [ ]:
if current_chunk_idx >= total_chunks:
    print("✅ All chunks have already been processed.")
else:
    start = current_chunk_idx * CHUNK_SIZE
    end = min(start + CHUNK_SIZE, total_examples)
    chunk = dataset[start:end]

    print("=" * 80)
    print(f"CHUNK {current_chunk_idx + 1} / {total_chunks}")
    print(f"Examples {start} → {end - 1}")
    print("=" * 80)
    print()

    print(json.dumps(chunk, indent=2, ensure_ascii=False))
    print()
    print("=" * 80)
    print("⬆️  Copy the JSON above and translate it using ChatGPT / Gemini.")
    print("=" * 80)

    # update progress
    with PROGRESS_FILE.open("w", encoding="utf-8") as f:
        json.dump({"current_chunk": current_chunk_idx + 1}, f)

## Notes on Translation

When translating a chunk:

- **Preserve the JSON structure exactly**
- Do NOT rename keys
- Do NOT remove or add fields
- Translate only natural language content:
  - question
  - context
  - reasoning
  - timeline
  - reflection
  - answer
- Keep all special tags unchanged:
  `<reasoning>`, `<timeline>`, `<reflection>`, `<answer>`

Once translated, the chunk can be appended to the target dataset
using the companion script.

# PROMPT for ChatGPT
You are translating a dataset used for training and evaluating a temporal reasoning model (TISER).

You will receive a JSON array of examples.
Your task is to translate the content from English to Italian.

IMPORTANT CONSTRAINTS (follow strictly):

1. Preserve the JSON structure EXACTLY.
   - Do not add or remove fields.
   - Do not rename keys.
   - Do not change the order of elements.
   - Output MUST be valid JSON.

2. Translate ONLY natural language text.
   - Translate fields such as:
     - "question"
     - "context"
     - "reasoning"
     - "timeline"
     - "reflection"
     - "answer"
   - Do NOT translate:
     - field names
     - IDs
     - dataset_name
     - dates, numbers, or years
     - proper names of people and places

3. Preserve ALL special tags and formatting EXACTLY.
   - Do NOT modify or translate:
     <reasoning>, </reasoning>
     <timeline>, </timeline>
     <reflection>, </reflection>
     <answer>, </answer>
   - Keep line breaks and indentation unchanged when possible.

4. Style requirements:
   - Use natural, fluent Italian.
   - Do NOT simplify or paraphrase.
   - Keep the same level of detail and reasoning.
   - Maintain temporal expressions faithfully.

5. Do NOT add explanations, comments, or notes.
   - Output ONLY the translated JSON.
   - No preamble, no markdown, no commentary.

Below is the JSON chunk to translate: